# Brownian Motion: First Simulations

I wrote this notebook because I wanted Brownian motion to feel like more than a definition in a lecture. The aim is small: define the process, simulate it on a finite grid, and check that the code has the variance scaling I expect.

A standard Brownian motion (or Wiener process) satisfies:

- $B_0=0$;
- for $0 \leq s < t$, the increment $B_t-B_s \sim \mathcal{N}(0,t-s)$;
- increments over disjoint time intervals are independent; and
- its paths are continuous almost surely.

The code below only observes the process at discrete time points. Joining those points with a line makes a useful plot, but it does not turn the simulation into the full continuous-time path.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 2026


def brownian_1d(n_steps=1_000, dt=0.01, rng=None):
    if n_steps <= 0:
        raise ValueError("n_steps must be positive")
    if dt <= 0:
        raise ValueError("dt must be positive")
    if rng is None:
        rng = np.random.default_rng()

    increments = rng.normal(0.0, np.sqrt(dt), size=n_steps)
    times = np.arange(n_steps + 1) * dt
    path = np.concatenate(([0.0], np.cumsum(increments)))
    return times, path


rng = np.random.default_rng(SEED)
t, B = brownian_1d(rng=rng)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t, B, linewidth=1)
ax.set(title="One simulated Brownian path", xlabel="Time", ylabel="$B_t$")
ax.grid(alpha=0.3)
plt.show()


## What changes in two dimensions?

Not much in the construction. I generate an independent Brownian coordinate for each axis, then plot one coordinate against the other instead of plotting position against time.


In [ ]:
def brownian_nd(n_dimensions, n_steps=1_000, dt=0.01, rng=None):
    if n_dimensions <= 0:
        raise ValueError("n_dimensions must be positive")
    if n_steps <= 0:
        raise ValueError("n_steps must be positive")
    if dt <= 0:
        raise ValueError("dt must be positive")
    if rng is None:
        rng = np.random.default_rng()

    increments = rng.normal(
        0.0, np.sqrt(dt), size=(n_steps, n_dimensions)
    )
    times = np.arange(n_steps + 1) * dt
    path = np.vstack((np.zeros(n_dimensions), np.cumsum(increments, axis=0)))
    return times, path


rng = np.random.default_rng(SEED + 1)
_, path_2d = brownian_nd(2, rng=rng)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(path_2d[:, 0], path_2d[:, 1], linewidth=1)
ax.scatter(0, 0, color="black", s=25, label="start")
ax.set(title="Two-dimensional Brownian path", xlabel="$B_t^{(1)}$", ylabel="$B_t^{(2)}$")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


The two-dimensional path is weirdly artistic, which made me want to see it in three dimensions. The maths is not deeper here; it is the same independent-increment construction with one more coordinate.

I used ChatGPT to help with the first version of the Matplotlib 3-D setup. I rewrote it around the same `brownian_nd` function so the simulation itself does not change between the two plots.


In [ ]:
rng = np.random.default_rng(SEED + 2)
_, path_3d = brownian_nd(3, rng=rng)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot(path_3d[:, 0], path_3d[:, 1], path_3d[:, 2], linewidth=1)
ax.scatter(0, 0, 0, color="black", s=25, label="start")
ax.set(
    title="Three-dimensional Brownian path",
    xlabel="$B_t^{(1)}$",
    ylabel="$B_t^{(2)}$",
    zlabel="$B_t^{(3)}$",
)
ax.legend()
plt.tight_layout()
plt.show()


## One path can be misleading

A single draw can wander almost anywhere, so I also want to see several paths and then check a property across many simulations.


In [ ]:
rng = np.random.default_rng(SEED + 3)
fig, ax = plt.subplots(figsize=(9, 4))

for _ in range(10):
    times, path = brownian_1d(rng=rng)
    ax.plot(times, path, alpha=0.7, linewidth=1)

ax.set(title="Ten Brownian paths", xlabel="Time", ylabel="$B_t$")
ax.grid(alpha=0.3)
plt.show()


## A basic numerical check

For standard Brownian motion, $B_T \sim \mathcal{N}(0,T)$. At $T=1$, a large set of terminal values should therefore have a sample mean near zero and sample variance near one. This does not prove the implementation is correct, but it should catch a broken scale or time grid.


In [ ]:
rng = np.random.default_rng(SEED + 4)
n_paths = 20_000
n_steps = 100
dt = 1.0 / n_steps
terminal_values = rng.normal(
    0.0, np.sqrt(dt), size=(n_paths, n_steps)
).sum(axis=1)

sample_mean = terminal_values.mean()
sample_variance = terminal_values.var(ddof=1)
print(f"sample mean: {sample_mean:.4f}")
print(f"sample variance: {sample_variance:.4f}")

assert abs(sample_mean) < 0.03
assert abs(sample_variance - 1.0) < 0.05


## What I took from this

The main bug in my first attempt was simple: I generated the first increment but labelled it as time zero. Prepending the known starting value and using `np.arange(n_steps + 1) * dt` makes the path and time grid agree.

The plots also made the difference between one realised path and a distribution much clearer. Brownian motion has no directional drift, while its uncertainty grows with time. That does not mean every financial price is Brownian or that randomness makes modelling pointless. It means any later price model needs extra assumptions, and I need to test those assumptions instead of reading too much into a nice-looking path.

Next I want to move the simulator into a small tested module, then use it to build geometric Brownian motion and compare the model assumptions with actual return data.
